# 02 — Prompt comparison avec MedGemma-4B-IT
Ce notebook est destiné à être exécuté par vos collègues sur une machine avec GPU.
Il charge le modèle `google/medgemma-4b-it`, lit les 3 prompts (baseline, improved, advanced), et évalue les performances sur les images.


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
from pathlib import Path
import json
import pandas as pd
import sys
sys.path.append(str(Path('..').resolve()))
from src.guardrails import validate_prediction

# --- 1. Chargement du modèle ---
model_id = "google/medgemma-4b-it"
print("Chargement du processeur...")
processor = AutoProcessor.from_pretrained(model_id)
print("Chargement du modèle (nécessite un GPU)...")
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"Modèle chargé sur : {model.device}")


In [ ]:
# --- 2. Chargement des prompts ---
prompts_dir = Path('../prompts')
prompts = {}
for mode in ['baseline', 'improved', 'advanced']:
    with open(prompts_dir / f"{mode}_prompt.txt", "r") as f:
        prompts[mode] = f.read()

print("Prompts chargés :", list(prompts.keys()))


In [ ]:
# --- 3. Fonction d'inférence avec MedGemma ---
def run_medgemma(image_path, prompt_text):
    image = Image.open(image_path).convert("RGB")
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    prompt_formatted = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(text=prompt_formatted, images=image, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=512, temperature=0.1)
    
    # Extraire uniquement la partie générée
    generated_text = processor.decode(output[0], skip_special_tokens=True)
    
    # Tenter de parser le JSON
    try:
        # Trouver les accolades pour extraire le JSON au cas où il y a du texte autour
        start_idx = generated_text.find('{')
        end_idx = generated_text.rfind('}') + 1
        if start_idx != -1 and end_idx != 0:
            json_str = generated_text[start_idx:end_idx]
            return json.loads(json_str), generated_text
        return None, generated_text
    except Exception as e:
        return None, generated_text


In [ ]:
# --- 4. Boucle d'évaluation sur les images ---
image_dir = Path('../data/sample_images/')
images = list(image_dir.glob('*.png'))[:10] # Limité à 10 images pour le test rapide, enlever le [:10] pour toutes les images

results = []

for img_path in images:
    print(f"Traitement de {img_path.name}...")
    for mode in ['baseline', 'improved', 'advanced']:
        prompt_text = prompts[mode]
        pred_json, raw_text = run_medgemma(img_path, prompt_text)
        
        if pred_json is None:
            results.append({
                "image": img_path.name,
                "mode": mode,
                "valid_json": False,
                "has_justification": False,
                "has_warning": False,
                "hallucination": False, # Non évaluable si JSON invalide
                "errors": "JSON Parsing Error"
            })
            continue
            
        # Validation du JSON généré
        is_valid, errors = validate_prediction(pred_json)
        
        has_justification = "justification" in pred_json and len(str(pred_json["justification"])) > 10
        has_warning = "warning" in pred_json and "Non destiné au diagnostic" in str(pred_json["warning"])
        hallucination = pred_json.get("predicted_class") not in ["normal", "suspected_opacity", "uncertain"]
        
        results.append({
            "image": img_path.name,
            "mode": mode,
            "valid_json": is_valid,
            "has_justification": has_justification,
            "has_warning": has_warning,
            "hallucination": hallucination,
            "errors": ", ".join(errors) if errors else "None"
        })

df = pd.DataFrame(results)


In [ ]:
# --- 5. Analyse des résultats ---
summary = df.groupby('mode').agg(
    valid_json_pct=('valid_json', lambda x: x.mean() * 100),
    justification_pct=('has_justification', lambda x: x.mean() * 100),
    warning_pct=('has_warning', lambda x: x.mean() * 100),
    hallucination_pct=('hallucination', lambda x: x.mean() * 100)
).reset_index()

summary['mode'] = pd.Categorical(summary['mode'], ["baseline", "improved", "advanced"])
summary = summary.sort_values("mode")
summary


## Ce que vous devriez observer
En exécutant ce notebook avec MedGemma-4B-IT, vous devriez voir que :
1. **Baseline** a souvent un taux de JSON valide plus faible (~75%) et omet des champs ou s'éloigne des classes permises (hallucination).
2. **Improved** gère mieux l'incertitude mais peut encore avoir des problèmes de format.
3. **Advanced** (avec l'instruction Chain-of-Thought et des règles strictes) devrait s'approcher des >=95% de validité JSON, inclure systématiquement les justifications et l'avertissement.
